In [25]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import os

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder

path = kagglehub.dataset_download("juhibhojani/house-price")
print("Path to dataset files:", path)
print(os.listdir(path))

Path to dataset files: C:\Users\MCA 41\.cache\kagglehub\datasets\juhibhojani\house-price\versions\1
['house_prices.csv']


In [26]:
file_path = os.path.join(path, "house_prices.csv")  # change name if different
df = pd.read_csv(file_path)

df.head()

df.info()
df.isnull().sum()
df = df.dropna()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 187531 entries, 0 to 187530
Data columns (total 21 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Index              187531 non-null  int64  
 1   Title              187531 non-null  object 
 2   Description        184508 non-null  object 
 3   Amount(in rupees)  187531 non-null  object 
 4   Price              169866 non-null  float64
 5   location           187531 non-null  object 
 6   Carpet Area        106858 non-null  object 
 7   Status             186916 non-null  object 
 8   Floor              180454 non-null  object 
 9   Transaction        187448 non-null  object 
 10  Furnishing         184634 non-null  object 
 11  facing             117298 non-null  object 
 12  overlooking        106095 non-null  object 
 13  Society            77853 non-null   object 
 14  Bathroom           186703 non-null  object 
 15  Balcony            138596 non-null  object 
 16  Ca

In [27]:
target_col = "Price"

if target_col not in df.columns:
    raise Exception(f"Target column '{target_col}' not found!")

X = df.drop(target_col, axis=1)
y = df[target_col]

for col in X.select_dtypes(include='object').columns:
    if X[col].nunique() > 100:   # threshold
        print(f"Dropping high-cardinality column: {col}")
        X = X.drop(col, axis=1)

for col in X.select_dtypes(include='object').columns:
    if X[col].nunique() < 20:
        X = pd.get_dummies(X, columns=[col], drop_first=True)
    else:
        le = LabelEncoder()
        X[col] = le.fit_transform(X[col].astype(str))

In [28]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Squared Error:", mse)
print("R2 Score:", r2)
coeff_df = pd.DataFrame(model.coef_, X.columns, columns=["Coefficient"])
print(coeff_df)

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [ ]:
plt.scatter(y_test, y_pred)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.show()